# Your First AI Agent: From Prompt to Action

This notebook is your first step into building AI agents. An agent can do more than just respond to a prompt — it can **take actions** to find information or get things done.

In this notebook, you'll:

- Install [Agent Development Kit (ADK)](https://google.github.io/adk-docs/)
- Configure your API key to use the Gemini model
- Build your first simple agent
- Run your agent and watch it use a tool (like Google Search) to answer a question

**Steps to obtain a Google Gemini API key (for Python use):**
*   **Open Google AI Studio**
    *   Go to **<https://aistudio.google.com>** and accept the terms if prompted.

*   **Create an API key**
    *   Click **“Get API key”** (or **Settings → API Keys**).
    *   Create a new key (optionally tie it to a specific project).

*   **Copy and store the key securely**
    *   Treat it like a password; don’t commit it to source control.

*   **(Optional) Set up a Google Cloud project**
    *   Recommended for production or higher usage (billing, quotas, monitoring).

*   **Add the key to your environment**
    *   e.g., set an environment variable: `GOOGLE_API_KEY=...`

*   **Verify with a simple test call**
    *   Run a short script to confirm the key works with a Gemini model.

## Section 1: Setup

### 1.1: Install dependencies

The Kaggle Notebooks environment includes a pre-installed version of the [google-adk](https://google.github.io/adk-docs/) library for Python and its required dependencies, so you don't need to install additional packages in this notebook.

To install and use ADK in your own Python development environment outside of this course, you can do so by running:

```
pip install google-adk
```

## CREATE and .env File

Here are the simplest ways to create a .env file in the same folder as your Jupyter notebook.

Run this cell in Jupyter:

        %%writefile my.env
        GOOGLE_API_KEY=your_api_key_here

What this does

1.Creates a file named .env
2.Saves it in the current working directory of the notebook

**Load the .env file**


        from dotenv import load_dotenv
        import os
        
        load_dotenv()
        
        api_key = os.getenv("GOOGLE_API_KEY")
        print(api_key)


### 1.2: Configure your Gemini API Key

This notebook uses the [Gemini API](https://ai.google.dev/gemini-api/docs), which requires authentication.

**1. Get your API key**

If you don't have one already, create an [API key in Google AI Studio](https://aistudio.google.com/app/api-keys).

**2. Add the key to Kaggle Secrets**

Next, you will need to add your API key to your Notebook.

**3. Authenticate in the notebook**

Run the cell below to complete authentication.

In [2]:
from __future__ import annotations

import asyncio
import os
import subprocess
import sys
from pathlib import Path
from typing import Optional

from dotenv import load_dotenv

MODEL_NAME = "gemini-2.5-flash-lite"
APP_NAME = "sample_agent_notebook"

In [3]:
from key import *

GEMINI_API_KEY = dict_keys.get("GEMINI_API_KEY")

# Make sure downstream libraries / subprocesses can see it.
os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

### 1.3: Import ADK components

Now, import the specific components you'll need from the Agent Development Kit and the Generative AI library. This keeps your code organized and ensures we have access to the necessary building blocks.

In [4]:
#%pip install google-adk

In [5]:
from google.adk.agents import Agent
from google.adk.models.google_llm import Gemini
from google.adk.runners import InMemoryRunner
from google.adk.tools import google_search
from google.genai import types

print("✅ ADK components imported successfully.")

✅ ADK components imported successfully.


In [6]:
import google.adk as adk
adk.__version__

'1.31.0'

### 1.4: Helper functions

We'll define some helper functions.

In [12]:
async def ask_agent(prompt: str):
    """
    Run one prompt through the agent.

    In a Jupyter notebook, use:
        response = await ask_agent("What is ADK from Google?")
        response

    In a normal Python script, use:
        asyncio.run(ask_agent("..."))
    """
    return await runner.run_debug(prompt)

#### Configure Retry Options

When working with LLMs, you may encounter transient errors like rate limits or temporary service unavailability. Retry options automatically handle these failures by retrying the request with exponential backoff.

In [13]:
retry_config=types.HttpRetryOptions(
    attempts=5,  # Maximum retry attempts
    exp_base=7,  # Delay multiplier
    initial_delay=1, # Initial delay before first retry (in seconds)
    http_status_codes=[429, 500, 503, 504] # Retry on these HTTP errors
)

---

## Section 2: Your first AI Agent with ADK

### 2.1 What is an AI Agent?

You've probably used an LLM like Gemini before, where you give it a prompt and it gives you a text response.

`Prompt -> LLM -> Text`

An AI Agent takes this one step further. An agent can think, take actions, and observe the results of those actions to give you a better answer.

`Prompt -> Agent -> Thought -> Action -> Observation -> Final Answer`

In this notebook, we'll build an agent that can take the action of searching Google. Let's see the difference!

### 2.2 Define your agent

Now, let's build our agent. We'll configure an `Agent` by setting its key properties, which tell it what to do and how to operate.

To learn more, check out the documentation related to [agents in ADK](https://google.github.io/adk-docs/agents/).

These are the main properties we'll set:

- **name** and **description**: A simple name and description to identify our agent.
- **model**: The specific LLM that will power the agent's reasoning. We'll use "gemini-2.5-flash-lite".
- **instruction**: The agent's guiding prompt. This tells the agent what its goal is and how to behave.
- **tools**: A list of [tools](https://google.github.io/adk-docs/tools/) that the agent can use. To start, we'll give it the `google_search` tool, which lets it find up-to-date information online.

#### 2.2.1 Define your agent: Blind and Powerless

Default Agents are cannot see and do anything...

In [14]:
root_agent = Agent(
    name="helpful_assistant",
    model=Gemini(
        model="gemini-2.5-flash-lite", #"gemini-3-flash-preview", #
        retry_options=retry_config
    ),
    description="A simple agent that can answer general questions.",
    instruction="You are a helpful assistant."
)

print("✅ Blind Agent defined.")

✅ Blind Agent defined.


In [15]:
runner = InMemoryRunner(agent=root_agent)

print("✅ Runner created.")

✅ Runner created.


In [16]:
#response = await runner.run_debug("What is Agent Development Kit from Google? What languages is the SDK available in?")
response = await ask_agent("What is the weather today in Dublin?")
#response


 ### Created new session: debug_session_id

User > What is the weather today in Dublin?
helpful_assistant > I'm sorry, I don't have access to real-time weather information. To get the most up-to-date weather for Dublin, I recommend checking a reliable weather website or app.


#### 2.2.1 Define your agent: Let's give them some powers....

In [17]:
root_agent = Agent(
    name="helpful_assistant",
    model=Gemini(
        model="gemini-2.5-flash-lite", #"gemini-3-flash-preview", #
        retry_options=retry_config
    ),
    description="A simple agent that can answer general questions.",
    instruction="You are a helpful assistant. Use Google Search for current info or if unsure.",
    tools=[google_search],
)

print("✅ Root Agent defined.")

✅ Root Agent defined.


### 2.3 Run your agent

Now it's time to bring your agent to life and send it a query. To do this, you need a [`Runner`](https://google.github.io/adk-docs/runtime/), which is the central component within ADK that acts as the orchestrator. It manages the conversation, sends our messages to the agent, and handles its responses.

**a. Create an `InMemoryRunner` and tell it to use our `root_agent`:**

In [18]:
runner = InMemoryRunner(agent=root_agent)

print("✅ Runner created.")

✅ Runner created.


👉 Note that we are using the Python Runner directly in this notebook. You can also run agents using ADK command-line tools such as `adk run`, `adk web`, or `adk api_server`. To learn more, check out the documentation related to [runtime in ADK](https://google.github.io/adk-docs/runtime/).

**b. Now you can call the `.run_debug()` method to send our prompt and get an answer.**

👉 This method abstracts the process of session creation and maintenance and is used in prototyping. We'll explore "what sessions are and how to create them" on Day 3.

In [20]:
#response = await runner.run_debug("What is Agent Development Kit from Google? What languages is the SDK available in?")
response = await ask_agent("What is ADK from Google?  What languages is the SDK available in?")
#response


 ### Continue session: debug_session_id

User > What is ADK from Google?  What languages is the SDK available in?
helpful_assistant > ADK, which stands for Agent Development Kit, is an open-source framework from Google designed to simplify the creation, debugging, and deployment of AI agents and multi-agent systems at an enterprise level. It provides developers with a comprehensive set of tools for the entire agent development lifecycle, from building agents and tools to orchestrating complex workflows and evaluating their performance. ADK aims to make it easier to build production-ready agent applications with flexibility and control, and it's the same framework used in Google products like Agentspace and the Google Customer Engagement Suite.

The ADK SDK is available in the following programming languages:
*   Python
*   TypeScript
*   Go
*   Java


In [35]:
#response

You can see a summary of ADK and its available languages in the response.

### 2.4 How does it work?

The agent performed a Google Search to get the latest information about ADK, and it knew to use this tool because:

1. The agent inspects and is aware of which tools it has available to use.
2. The agent's instructions specify the use of the search tool to get current information or if it is unsure of an answer.

The best way to see the full, detailed trace of the agent's thoughts and actions is in the **ADK web UI**, which we'll set up later in this notebook.

And we'll cover more detailed workflows for logging and observability later in the course.

### 2.5 Your Turn!

This is your chance to see the agent in action. Ask it a question that requires current information.

Try one of these, or make up your own:

- What's the weather in London?
- Who won the last soccer world cup?
- What new movies are showing in theaters now?

In [21]:
response = await ask_agent("What's the weather in Dublin? Any chance to experience a snowfall over the next 5 days?")


 ### Continue session: debug_session_id

User > What's the weather in Dublin? Any chance to experience a snowfall over the next 5 days?
helpful_assistant > The weather in Dublin is currently partly sunny with a temperature of 60°F (16°C). There is a 10% chance of rain today, with light rain expected during the day and at night. The humidity is around 61%.

Looking ahead to the next five days, the chance of snow in Dublin is very low. Forecasts indicate a 0% chance of snow for the next several days, including today (April 30th), May 1st, and May 2nd. On May 5th, there is a slight chance of snow at 3%, increasing to 9% on May 6th. However, temperatures are generally expected to remain mild, with highs in the 50s and 60s Fahrenheit and lows in the 40s Fahrenheit. The forecast mainly calls for clouds and a chance of rain showers, with some of the rain showers potentially being heavy.


In [37]:
response = await ask_agent("What was my last question?")


 ### Continue session: debug_session_id

User > What was my last question?
helpful_assistant > Your last question was: "What's the weather in Dublin? Any chance to experience a snowfall over the next 5 days?"


---
## Section 3: Different Approach

### Overview

Here we have created an agent on the fly. However, they should be encapsulated within *.py* files

In [13]:
# -----------------------------------------------------------------------------
# 1) Create the sample_agent folder without using `adk create`
# -----------------------------------------------------------------------------

def create_sample_agent_folder(
    agent_name: str = "sample_agent",
    model_name: str = MODEL_NAME,
    api_key: Optional[str] = None,
    overwrite: bool = False,
) -> Path:
    """
    Create the same minimal scaffold that `adk create` would generate:
    - sample_agent/.env
    - sample_agent/__init__.py
    - sample_agent/agent.py

    This avoids notebook/Windows console encoding issues that can happen when
    running `adk create` from Jupyter.
    """
    api_key = api_key or os.getenv("GEMINI_API_KEY")
    target = Path(agent_name)
    target.mkdir(parents=True, exist_ok=True)

    files = {
        target / ".env": (
            "GOOGLE_GENAI_USE_VERTEXAI=0\n"
            f"GOOGLE_API_KEY={api_key or ''}\n"
        ),
        target / "__init__.py": "from . import agent\n",
        target / "agent.py": (
            "from google.adk.agents.llm_agent import Agent\n\n"
            "root_agent = Agent(\n"
            f"    model='{model_name}',\n"
            "    name='root_agent',\n"
            "    description='A helpful assistant for user questions.',\n"
            "    instruction='Answer user questions to the best of your knowledge',\n"
            ")\n"
        ),
    }

    for file_path, text in files.items():
        if file_path.exists() and not overwrite:
            continue
        file_path.write_text(text, encoding="utf-8")

    return target.resolve()

# -----------------------------------------------------------------------------
# 2) Optional: launch the ADK web UI locally
# -----------------------------------------------------------------------------
def start_adk_web(agent_parent_dir: str = ".", port: int = 8000) -> subprocess.Popen:
    """
    Launch `adk web` as a background process.

    IMPORTANT:
    - Run this from the *parent directory* that contains `sample_agent/`.
    - If `adk` is not on PATH inside your notebook kernel, you may need to run
      it from a terminal instead.
    - For a local Jupyter notebook, the UI is typically available at:
          http://127.0.0.1:8000
    """
    env = os.environ.copy()
    env.setdefault("PYTHONUTF8", "1")
    env.setdefault("PYTHONIOENCODING", "utf-8")

    process = subprocess.Popen(
        ["adk", "web", "--port", str(port)],
        cwd=str(Path(agent_parent_dir).resolve()),
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    return process

async def demo() -> None:
    print("Testing the notebook-friendly ADK agent...\n")

    response = await ask_agent(
        "What is Agent Development Kit from Google? What languages is the SDK available in?"
    )
    print(response)

    print("\nCreating local scaffold...\n")
    sample_path = create_sample_agent_folder("sample_agent_v0", overwrite=False)
    print(f"Scaffold ready at: {sample_path}")
    print("You can now run `adk web` from the parent directory of sample_agent/.")    

In [14]:
# sample_path = create_sample_agent_folder("sample_agent_v0", overwrite=False)
# print(f"Scaffold ready at: {sample_path}")

In [15]:
# start_adk_web()

## Section 4: Using different models

This version is fully runnable as-is. It includes:

- an OpenAI/ChatGPT model via LiteLlm(model="openai/gpt-4o-mini"),
- a simple tool (calculate_square) wrapped with FunctionTool, 
- an InMemoryRunner, a session, and a sample user query using types.Content(...)


In [ ]:
#%pip install "google-adk[litellm]" litellm openai python-dotenv

In [ ]:
from google.adk.agents import Agent
from google.adk.runners import InMemoryRunner
from google.adk.models import LiteLlm
from google.adk.tools import FunctionTool
from google.genai import types

In [18]:
OPENAI_API_KEY = "XOXOXOXOXOXOX"

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise EnvironmentError(
        "OPENAI_API_KEY is not set. Please add it to your .env file or environment variables."
    )

print("OPENAI_API_KEY loaded successfully.")

Example: A new tool 

In [ ]:
model = LiteLlm(model="openai/gpt-4o-mini")

root_agent = Agent(
    name="helpful_assistant",
    model=model,
    description="A simple agent that can answer general questions.",
    instruction=(
        "You are a helpful assistant. "
        "If needed, use the available tools to answer accurately."
    )
)

In [ ]:
app_name = "helpful_assistant_app"
user_id = "user_001"

runner = InMemoryRunner(agent=root_agent, app_name=app_name)

session = await runner.session_service.create_session(
    app_name=app_name,
    user_id=user_id,
)

print("Session created:", session.id)

In [ ]:
query = "What is the square of 12?"

In [ ]:
await runner.run_debug(query)